# Ước lượng độ sâu tĩnh mạch từ ảnh phản xạ 850 nm — notebook tự chứa (chạy local)

- **Mọi code nằm trong notebook**; notebook không gọi script nào của repo. Repo chỉ cung cấp **dữ liệu và mô hình**, đọc qua đường dẫn ở ô 2.
- Dữ liệu mô phỏng (Monte Carlo) và mô hình đã có sẵn: không chạy lại mô phỏng, không cần GPU, không cần token.
- Kết quả ảnh tay thật ghi vào `real_hand/results` và `real_hand/figs` của repo (đã có trong .gitignore).

| Phần | Ô |
|---|---|
| 0. Chuẩn bị: thư viện, đường dẫn, kiểm file khóa | 1–3 |
| Định nghĩa dùng chung: mạng CNN, xử lý profile | 4–5 |
| 1. Da đồng nhất (thử nghiệm 1) | 6–7 |
| 2. Da nhiều lớp (thử nghiệm 2) | 8–10 |
| 3. Ảnh tay thật | 11–17 |
| 4. Nơi lưu kết quả | 18 |

**Bộ test khóa chỉ được chấm một lần: notebook chỉ đọc kết quả của nó, không chấm lại.**

## 0. Chuẩn bị

In [ ]:
# Ô 1 — Thư viện (cài vào kernel này nếu thiếu) và môi trường
import sys, os, json, csv, hashlib, platform, subprocess, importlib.util
from pathlib import Path
need = {"numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib", "torch": "torch", "PIL": "pillow"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Cài:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage as ndi
from IPython.display import display, HTML
print("Python", sys.version.split()[0], "|", platform.platform())
print("numpy", np.__version__, "| torch", torch.__version__)

In [ ]:
# Ô 2 — Đường dẫn dữ liệu (chỉ đọc dữ liệu/mô hình từ đây; muốn đổi chỗ thì sửa ở ô này)
REPO_URL = "https://github.com/Hoang-Minh-Nghia/vien_enh.git"

def find_repo(start):
    for p in [start, *start.parents]:
        if (p / "exp1_homogeneous").is_dir() and (p / "exp2_layered").is_dir():
            return p
    return None

REPO = Path(os.environ["VEIN_REPO"]).resolve() if os.environ.get("VEIN_REPO") else find_repo(Path.cwd().resolve())
if REPO is None:                      # notebook nằm ngoài repo: tải dữ liệu về bằng git (đăng nhập git sẵn có của máy)
    REPO = Path.cwd() / "vien_enh"
    if not REPO.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)

E1, E2 = REPO / "exp1_homogeneous", REPO / "exp2_layered"
PATH = {
    # thử nghiệm 1 — da đồng nhất
    "exp1_dataset":     E1 / "03_dataset/data/dataset_FK/dataset.npz",
    "exp1_models":      E1 / "04_training/results/FK/models",              # cnn_F/K_seed{42,43,44}.pt
    # thử nghiệm 2 — da nhiều lớp
    "exp2_data":        E2 / "03_dataset/data",                            # manifest*.json, <split>/<id>.npz
    "exp2_variants":    E2 / "04_training/results/variants",               # hetero/wide/... *.pt
    "exp2_layered_cnn": E2 / "04_training/results/layered_cnn/models",     # F_layered_seed*.pt
    "exp2_locked":      E2 / "06_final_test/results/locked_test",          # eval.json, predictions.npz
    # kiểm toàn vẹn
    "frozen_hashes":    REPO / "docs/FROZEN_FILES.sha256",
    # ảnh tay thật (KHÔNG có trong repo) và nơi ghi kết quả của nó
    "real_data":        Path(os.environ.get("VEIN_REAL_DATA", r"C:\luanvan\deliver\My_DATASET")),
    "real_out":         REPO / "real_hand/results/depth_points",
    "real_figs":        REPO / "real_hand/figs",
}
for k, v in PATH.items():
    state = "nơi ghi" if k in ("real_out", "real_figs") else ("có" if v.exists() else "THIẾU")
    print(f"{k:17s} {state:8s} {v}")

In [ ]:
# Ô 3 — Kiểm toàn vẹn: các file đã khóa (cấu hình + lõi mô phỏng) còn đúng từng byte
ok = True
for line in PATH["frozen_hashes"].read_text().splitlines():
    h, rel = line.split(maxsplit=1)
    good = hashlib.sha256((REPO / rel).read_bytes()).hexdigest() == h
    ok &= good
    print("OK  " if good else "SAI ", rel)
print("\nTất cả file khóa đúng" if ok else "\nCÓ FILE BỊ THAY ĐỔI — không dùng kết quả")

## Định nghĩa dùng chung

In [ ]:
# Ô 4 — Hai kiến trúc mạng (giống hệt lúc huấn luyện) và hàm nạp mô hình
class DepthCNN(nn.Module):
    """CNN của thử nghiệm 1 (F, K, F_layered): 3 tích chập 16/32/64, gộp trung bình, 1 đầu ra độ sâu."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv1d(1, 16, 5, padding=2), nn.ReLU(),
                                      nn.Conv1d(16, 32, 5, padding=2), nn.ReLU(),
                                      nn.Conv1d(32, 64, 5, padding=2), nn.ReLU(),
                                      nn.AdaptiveAvgPool1d(1), nn.Flatten())
        self.head = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):
        return self.head(self.features(x)).squeeze(-1)


class WideCNN(nn.Module):
    """CNN của thử nghiệm 2: tích chập giãn (dilation 1..16, vùng nhìn ~30 mm), BatchNorm, gộp trung bình + max."""
    def __init__(self, n_out=1):
        super().__init__()
        ch, layers, c_in = 48, [], 1
        for d in (1, 2, 4, 8, 16):
            layers += [nn.Conv1d(c_in, ch, 5, padding=2 * d, dilation=d), nn.BatchNorm1d(ch), nn.ReLU()]
            c_in = ch
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Linear(2 * ch, 64), nn.ReLU(), nn.Linear(64, n_out))

    def forward(self, x):
        h = self.features(x)
        return self.head(torch.cat([h.mean(-1), h.amax(-1)], 1))


def build(variant):
    """variant: base/sched = DepthCNN; wide = 1 đầu ra; multi = 4 đầu ra; hetero = (độ sâu, log σ)."""
    return DepthCNN() if variant in ("base", "sched") else WideCNN({"wide": 1, "multi": 4, "hetero": 2}[variant])


def load_net(path, variant=None):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    v = variant or ck.get("variant", "base")
    m = build(v); m.load_state_dict(ck["model"])
    return v, m.eval()


def outputs(model, X):
    """X: (n, 128) → mảng (n, k) đầu ra của mạng."""
    with torch.no_grad():
        o = model(torch.from_numpy(np.asarray(X, np.float32)[:, None, :]))
    return (o[:, None] if o.dim() == 1 else o).numpy()

torch.set_num_threads(8)
print("ok")

In [ ]:
# Ô 5 — Xử lý profile: lưới 128 bin, dịch ngang + nhiễu camera, đọc dữ liệu da nhiều lớp
EDGES = np.linspace(-16, 16, 129)                 # 128 bin × 0,25 mm — đầu vào của mạng
XC = (EDGES[:-1] + EDGES[1:]) / 2


class Binner:
    """Gộp profile mịn (bước 0,1 mm) vào 128 bin, có dịch ngang `shift` (mm) — bảo toàn tích phân."""
    def __init__(self, x):
        dx = float(np.median(np.diff(x)))
        self.src = np.r_[x - dx / 2, x[-1] + dx / 2]
        self.w = np.diff(EDGES)

    def __call__(self, P, shifts):
        cum = np.concatenate([np.zeros((len(P), 1)), np.cumsum(P * np.diff(self.src), 1)], 1)
        out = np.empty((len(P), len(EDGES) - 1))
        for i, sh in enumerate(shifts):
            out[i] = np.diff(np.interp(EDGES - sh, self.src, cum[i])) / self.w
        return out


def augment(binner, P, rng, sigma=None):
    """Dịch ngang U(−2, 2) mm rồi cộng nhiễu trắng σ (cố định, hoặc log-đều 0,001–0,01 nếu sigma=None)."""
    sh = rng.uniform(-2, 2, len(P))
    X = binner(P, sh)
    sa = np.exp(rng.uniform(np.log(0.001), np.log(0.01), len(P))) if sigma is None else np.full(len(P), sigma)
    X += rng.normal(0, 1, X.shape) * sa[:, None]
    return X.astype(np.float32)


def load_split(split, manifest):
    """Đọc một bộ da nhiều lớp: mỗi file = 1 làn da, nhiều mạch; profile = profiles_y_mean / n0_scalar."""
    m = json.loads((PATH["exp2_data"] / manifest).read_text(encoding="utf-8"))
    P, meta = [], []
    for cm in m["splits"][split]["configs"]:
        f = PATH["exp2_data"] / split / f"{cm['id']}.npz"
        if not f.exists():
            continue
        z = np.load(f)
        c = json.loads(str(z["config_json"])); assert c == cm
        x, s = z["x_mm"], float(z["n0_scalar"])
        for v, p in zip(c["vessels"], z["profiles_y_mean"]):
            P.append(p.astype(np.float64) / s)
            meta.append({"config": c["id"], **v, **c["layers"]})
    return x, np.array(P), meta

print("ok")

## 1. Da đồng nhất (thử nghiệm 1)

In [ ]:
# Ô 6 — Dự đoán 600 cảnh test bằng 6 mô hình F/K (3 seed)
z = np.load(PATH["exp1_dataset"], allow_pickle=False)
test = z["split"] == "test"
X1, y1, mua1, sid1 = z["X_F"][test].astype(np.float32), z["y_mm"][test], z["mua_F"][test], z["scene_id"][test]
pred1 = {}
for seed in (42, 43, 44):
    for br in "FK":                                  # cả hai nhánh đều được chấm trên đầu vào F
        _, m = load_net(PATH["exp1_models"] / f"cnn_{br}_seed{seed}.pt", "base")
        pred1[br, seed] = outputs(m, X1)[:, 0]
print(f"{len(y1)} cảnh test, độ sâu {y1.min():.2f}–{y1.max():.2f} mm, mức hấp thụ {sorted({round(float(a), 3) for a in mua1})} mm⁻¹")

In [ ]:
# Ô 7 — MAE theo nhóm hấp thụ (trung bình 3 seed) và sai số theo độ sâu
groups = {"Hấp thụ yếu 0,15–0,30 (chỉ số chính)": mua1 < 0.4, "Hấp thụ chuẩn 0,508": np.isclose(mua1, 0.508),
          "Hấp thụ cao 1,0": mua1 > 0.9, "Toàn bộ test": np.ones(len(y1), bool)}
print(f"{'Nhóm':40s} {'MAE F':>7s} {'MAE K':>7s}")
for name, msk in groups.items():
    mf = np.mean([np.abs(pred1["F", s][msk] - y1[msk]).mean() for s in (42, 43, 44)])
    mk = np.mean([np.abs(pred1["K", s][msk] - y1[msk]).mean() for s in (42, 43, 44)])
    print(f"{name:40s} {mf:7.3f} {mk:7.3f}")
edges = np.arange(0.5, 5.01, 0.5); mid = (edges[:-1] + edges[1:]) / 2
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for a, br in zip(ax, "FK"):
    for mu, col in zip((0.15, 0.30, 0.508, 1.0), ("C0", "C1", "C2", "C3")):
        e = [np.mean([np.abs(pred1[br, s] - y1)[np.isclose(mua1, mu) & (y1 >= lo) & (y1 < hi)].mean() for s in (42, 43, 44)])
             for lo, hi in zip(edges[:-1], edges[1:])]
        a.plot(mid, e, "o-", color=col, label=f"μa {mu} mm⁻¹")
    a.set_title(f"CNN {br}"); a.set_xlabel("Độ sâu (mm)"); a.grid(alpha=.3)
ax[0].set_ylabel("MAE (mm)"); ax[1].legend(); plt.tight_layout(); plt.show()

## 2. Da nhiều lớp (thử nghiệm 2)

In [ ]:
# Ô 8 — Chấm mọi mô hình trên validation_ext (90 làn da mới) ở 3 mức nhiễu camera
x2, P2, meta2 = load_split("validation_ext", "manifest_ext.json")
b2 = Binner(x2)
sig_mc = float(np.median(b2(P2, np.zeros(len(P2)))[:, np.abs(XC) > 12].std(1)))    # nhiễu MC sẵn có
y2 = np.array([m["depth_mm"] for m in meta2])
models2 = {f.stem: load_net(f) for f in sorted(PATH["exp2_variants"].glob("*.pt"))}
for s in (42, 43, 44):
    models2[f"t5_F_layered_seed{s}"] = load_net(PATH["exp2_layered_cnn"] / f"F_layered_seed{s}.pt", "base")
    models2[f"F_locked_seed{s}"] = load_net(PATH["exp1_models"] / f"cnn_F_seed{s}.pt", "base")
res2 = {}
for k, tot in enumerate((0.005, 0.007, 0.010)):
    add = float(np.sqrt(max(tot ** 2 - sig_mc ** 2, 0.0)))                          # nhiễu thêm để đạt σ tổng
    rng = np.random.default_rng(700 + k)
    X = np.concatenate([augment(b2, P2, rng, sigma=add) for _ in range(4)]); yy = np.tile(y2, 4)
    pr = {tag: outputs(m, X)[:, 0] for tag, (v, m) in models2.items()}
    fam = {}
    for tag in pr:
        fam.setdefault(tag.rsplit("_seed", 1)[0], []).append(tag)
    for g, tags in fam.items():
        res2.setdefault(g, []).append(float(np.abs(np.mean([pr[t] for t in tags], 0) - yy).mean()))
print(f"σ_mc = {sig_mc:.4f}\n{'Mô hình':16s} MAE (mm), TB 3 mức nhiễu")
for g, v in sorted(res2.items(), key=lambda kv: np.mean(kv[1])):
    print(f"{g:16s} {np.mean(v):8.4f}")

In [ ]:
# Ô 9 — Kết quả bộ test khóa — CHỈ ĐỌC (bộ test đã mở một lần)
e = json.loads((PATH["exp2_locked"] / "eval.json").read_text(encoding="utf-8"))
p, ci = e["primary_mean_over_3_levels"], e["primary_hetero_ci95"]
label = {"hetero": "hetero WideCNN (mô hình cuối)", "wide": "WideCNN", "F_layered": "CNN da nhiều lớp",
         "F_locked": "CNN da đồng nhất (thử nghiệm 1)", "B3_homog_joint": "Khớp vật lý đồng nhất (B3)",
         "constant_train_mean": "Đoán hằng số"}
print(f"{e['n_vessels']} mạch trên {e['n_skins']} làn da chưa từng dùng\n")
for k, v in label.items():
    print(f"{v:36s} {p[k]:.3f}" + (f"   KTC 95 % {ci[0]:.3f}–{ci[1]:.3f}" if k == "hetero" else ""))
print(f"\n{'Mức nhiễu':10s} {'MAE':>6s} {'bias':>7s} {'phủ ±1,96σ':>11s}")
for name, lev in e["levels"].items():
    h = lev["hetero"]
    print(f"{name:10s} {h['mae']:6.3f} {h['bias']:+7.3f} {100 * h['coverage']['95']:10.1f} %")

In [ ]:
# Ô 10 — Biểu đồ bộ test khóa
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
keys = ["hetero", "F_layered", "constant_train_mean", "F_locked", "B3_homog_joint"]
ax[0].barh([label[k] for k in keys][::-1], [p[k] for k in keys][::-1], color=["#999"] * 4 + ["#1f77b4"])
ax[0].errorbar(p["hetero"], 0, xerr=[[p["hetero"] - ci[0]], [ci[1] - p["hetero"]]], fmt="none", ecolor="k", capsize=4)
ax[0].set_xlabel("MAE (mm)"); ax[0].set_title("Bộ test khóa: 40 làn da mới")
lv = [k for k in e["levels"] if k != "mc_only"]; sig = [float(k.split("_")[1]) for k in lv]
ax[1].plot(sig, [e["levels"][k]["hetero"]["mae"] for k in lv], "o-", label="MAE")
ax[1].plot(sig, [e["levels"][k]["hetero"]["rmse"] for k in lv], "s--", label="RMSE")
ax[1].set_xlabel("Nhiễu tổng mỗi bin (σ)"); ax[1].set_ylabel("mm"); ax[1].set_title("hetero theo mức nhiễu camera")
ax[1].legend(); ax[1].grid(alpha=.3); plt.tight_layout(); plt.show()

## 3. Ảnh tay thật

**Ảnh tay thật không có trong repo.** Đây là dữ liệu người thật: chỉ dùng khi có sự đồng ý của người được chụp; không commit, không chia sẻ kết quả cho người ngoài.

Thư mục `PATH["real_data"]` (ô 2) cần có `images/<người>_<L|R>_<số>.png` (ảnh xám 850 nm) và `masks/` cùng tên (mặt nạ mạch, khác 0 = mạch).
Không có ground truth: các con số dưới đây là **dự đoán của mô hình**, không phải số đo.

In [ ]:
# Ô 11 — Thông số đo và kiểm dữ liệu ảnh (không in tên file)
MM_PER_PX = 156.6 / 752          # tỉ lệ hệ chụp: khung 156,6 × 116,6 mm trên 752 × 560 pixel
DARK = 0.0                       # mức tối của camera trừ đi trước khi chia nền
CELL, AVG_MM, EXCL_MM, EDGE_MM = 14, 1.5, 6.0, 1.0
DS = PATH["real_data"]
imgs = sorted((DS / "images").glob("*.png"))
masks = {m.name for m in (DS / "masks").glob("*.png")}
paired = [f for f in imgs if f.name in masks]
print(f"{len(imgs)} ảnh, {len(masks)} mặt nạ, {len(paired)} cặp khớp tên")
if paired:
    w, h = Image.open(paired[0]).size
    print(f"Kích thước {w} × {h} px → khung đo {w * MM_PER_PX:.1f} × {h * MM_PER_PX:.1f} mm")

In [ ]:
# Ô 12 — Các bước đo trên một ảnh: vùng bàn tay, điểm tâm mạch, hướng mạch, lát cắt ngang và chia nền
def otsu(v):
    h, e = np.histogram(v, 256, (0, 256)); p = h / h.sum(); c = (e[:-1] + e[1:]) / 2
    w = np.cumsum(p); m = np.cumsum(p * c)
    return c[np.argmax((m[-1] * w - m) ** 2 / (w * (1 - w) + 1e-12))]

def hand_roi(img):
    """Chỉ giữ bàn tay: ngưỡng Otsu trên ảnh làm mờ, thành phần lớn nhất, lấp lỗ, co vào EDGE_MM."""
    g = ndi.gaussian_filter(img, 4)
    h = ndi.binary_opening(g > otsu(g), iterations=5)
    lab, n = ndi.label(h)
    if n == 0:
        return h
    h = ndi.binary_fill_holes(lab == 1 + int(np.argmax(ndi.sum(h, lab, range(1, n + 1)))))
    return ndi.binary_erosion(h, iterations=max(1, int(round(EDGE_MM / MM_PER_PX))))

def centre_points(mask):
    """Đỉnh của biến đổi khoảng cách = đường tâm mạch; giữ 1 điểm mỗi ô CELL×CELL pixel."""
    dt = ndi.distance_transform_edt(mask)
    ys, xs = np.nonzero((dt >= ndi.maximum_filter(dt, 3)) & (dt >= 2))
    keep, seen = [], set()
    for y, x in sorted(zip(ys, xs), key=lambda q: -dt[q]):
        if (y // CELL, x // CELL) not in seen:
            seen.add((y // CELL, x // CELL)); keep.append((y, x))
    return keep, dt

def directions(mask):
    """Hướng vuông góc với mạch, từ tensor cấu trúc của mặt nạ làm mờ."""
    m = ndi.gaussian_filter(mask.astype(float), 3)
    gy, gx = ndi.sobel(m, 0), ndi.sobel(m, 1)
    Jxx, Jyy, Jxy = (ndi.gaussian_filter(a, 6) for a in (gx * gx, gy * gy, gx * gy))
    ang = 0.5 * np.arctan2(2 * Jxy, Jxx - Jyy)
    return np.cos(ang), np.sin(ang)

def profile(img, mask, hand, y, x, nx, ny):
    """Lát cắt ±16 mm qua (x, y), trung bình 7 đường trong ±AVG_MM dọc mạch; chia cho nền bậc 2 khớp ở |t| > EXCL_MM."""
    t_mm = np.arange(-16, 16.001, 0.125); s_mm = np.linspace(-AVG_MM, AVG_MM, 7)
    T, S = np.meshgrid(t_mm / MM_PER_PX, s_mm / MM_PER_PX)
    X = x + T * nx - S * ny; Y = y + T * ny + S * nx
    if X.min() < 0 or Y.min() < 0 or X.max() > img.shape[1] - 1 or Y.max() > img.shape[0] - 1:
        return None
    if ndi.map_coordinates(hand.astype(float), [Y, X], order=0).mean() < 0.999:
        return None                                                    # lát cắt chạm ra ngoài bàn tay
    p = ndi.map_coordinates(img, [Y, X], order=1).mean(0) - DARK
    mv = ndi.map_coordinates(mask.astype(float), [Y[3], X[3]], order=0) > 0.5
    c = len(t_mm) // 2
    if not mv[c]:
        return None
    lo = hi = c
    while lo > 0 and mv[lo - 1]: lo -= 1
    while hi < len(mv) - 1 and mv[hi + 1]: hi += 1
    width_mm = (hi - lo + 1) * 0.125
    others = mv.copy(); others[lo:hi + 1] = False
    oth_d = ndi.binary_dilation(others, iterations=8)                  # mạch khác + 1 mm
    fit = ~oth_d & (np.abs(t_mm) > EXCL_MM)
    if fit.sum() < 40 or (fit & (t_mm < 0)).sum() < 15 or (fit & (t_mm > 0)).sum() < 15:
        return None
    base = np.polyval(np.polyfit(t_mm[fit], p[fit], 2), t_mm)          # nền tại từng vị trí
    if base.min() <= 5:
        return None
    r = p / base
    r[oth_d & (np.abs(t_mm) > width_mm)] = 1.0
    B = np.array([r[(t_mm >= a) & (t_mm < b)].mean() for a, b in zip(EDGES[:-1], EDGES[1:])])
    return B.astype(np.float32), width_mm, float(1 - r[lo:hi + 1].min())

HETERO = [load_net(PATH["exp2_variants"] / f"hetero_seed{s}.pt")[1] for s in (42, 43, 44)]   # mô hình cuối

def predict_depth(profiles):
    """Ensemble 3 seed: độ sâu trung bình và σ (gộp độ bất định từng mạng + chênh giữa các mạng)."""
    o = [outputs(m, profiles) for m in HETERO]
    mu = np.array([q[:, 0] for q in o]); sg = np.exp(np.clip([q[:, 1] for q in o], -6, 3))
    d = mu.mean(0)
    return d, np.sqrt((sg ** 2 + mu ** 2).mean(0) - d ** 2)
print("ok")

In [ ]:
# Ô 13 — Đo độ sâu trên mọi ảnh (vài phút trên CPU) → real_hand/results/depth_points/points_cal.csv
torch.set_num_threads(2)
rows, profs = [], []
for f in paired:
    img = np.asarray(Image.open(f), dtype=float)
    hand = hand_roi(img)
    mask = (np.asarray(Image.open(DS / "masks" / f.name)) > 0) & hand
    pts, _ = centre_points(mask)
    cx, cy = directions(mask)
    for (y, x) in pts:
        o = profile(img, mask, hand, y, x, cx[y, x], cy[y, x])
        if o is not None:
            person, side = f.stem.rsplit("_", 2)[:2]
            rows.append({"image": f.stem, "person": person, "side": side, "x": int(x), "y": int(y),
                         "mask_width_mm": o[1], "contrast": o[2]}); profs.append(o[0])
d, sd = predict_depth(np.array(profs))
for r, a, b in zip(rows, d, sd):
    r["depth_mm"], r["sigma_mm"] = float(a), float(b)
PATH["real_out"].mkdir(parents=True, exist_ok=True)
with open(PATH["real_out"] / "points_cal.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
np.save(PATH["real_out"] / "profiles_cal.npy", np.array(profs))
torch.set_num_threads(8)
print(f"{len(rows)} điểm đo trên {len({r['image'] for r in rows})} ảnh")

In [ ]:
# Ô 14 — Thống kê độ sâu dự đoán (ẩn danh: không in tên file)
per_img = {}
for r in rows:
    per_img.setdefault(r["image"], []).append(r["depth_mm"])
med_img = np.array([np.median(v) for v in per_img.values()])
print(f"Độ sâu trung vị {np.median(d):.2f} mm (IQR {np.percentile(d, 25):.2f}–{np.percentile(d, 75):.2f}); "
      f"σ dự báo trung vị {np.median(sd):.2f} mm")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].hist(d, bins=40, color="#1f77b4"); ax[0].set_xlabel("Độ sâu dự đoán (mm)"); ax[0].set_ylabel("Số điểm"); ax[0].set_title("Mọi điểm đo")
ax[1].hist(med_img, bins=30, color="#ff7f0e"); ax[1].set_xlabel("Trung vị mỗi ảnh (mm)"); ax[1].set_ylabel("Số ảnh"); ax[1].set_title("Mỗi ảnh")
plt.tight_layout(); plt.show()

In [ ]:
# Ô 15 — So mô hình da nhiều lớp (hetero) với mô hình da đồng nhất (F_locked) trên cùng profile thật
Xr = np.array(profs)
dF = np.mean([outputs(load_net(PATH["exp1_models"] / f"cnn_F_seed{s}.pt", "base")[1], Xr)[:, 0] for s in (42, 43, 44)], 0)
print(f"Trung vị: hetero {np.median(d):.2f} mm | F_locked {np.median(dF):.2f} mm")
print(f"Chênh (F_locked − hetero): trung vị {np.median(dF - d):+.2f} mm, IQR {np.percentile(dF - d, 25):+.2f} … {np.percentile(dF - d, 75):+.2f} mm")
plt.figure(figsize=(5, 5)); plt.plot(d, dF, ".", ms=2, alpha=.3); plt.plot([0, 6], [0, 6], "k--")
plt.xlabel("hetero (da nhiều lớp), mm"); plt.ylabel("F_locked (da đồng nhất), mm"); plt.title("Cùng profile ảnh thật"); plt.show()
print("Không có độ sâu thật: chênh lệch chỉ cho biết hai mô hình khác nhau bao nhiêu, không nói cái nào đúng.")

In [ ]:
# Ô 16 — Bản đồ 3D: làm mảnh mặt nạ thành đường tâm, đo ở mọi điểm, nội suy điều hòa chỗ thiếu, làm trơn dọc mạch
from scipy import sparse
from scipy.sparse import csgraph
from scipy.sparse.linalg import spsolve
MIN_COMP, SMOOTH_IT, GAP_MM = 15, 120, 1.0

def thin(mask):
    """Làm mảnh Zhang–Suen (vector hóa) → đường tâm rộng 1 pixel."""
    im = np.pad(mask.astype(np.uint8), 1)
    while True:
        changed = False
        for step in (0, 1):
            P = im
            p2, p3, p4, p5 = P[:-2, 1:-1], P[:-2, 2:], P[1:-1, 2:], P[2:, 2:]
            p6, p7, p8, p9 = P[2:, 1:-1], P[2:, :-2], P[1:-1, :-2], P[:-2, :-2]
            nb = [p2, p3, p4, p5, p6, p7, p8, p9]
            B = sum(n.astype(int) for n in nb)
            A = sum(((nb[i] == 0) & (nb[(i + 1) % 8] == 1)).astype(int) for i in range(8))
            c = P[1:-1, 1:-1] == 1
            if step == 0:
                m = c & (B >= 2) & (B <= 6) & (A == 1) & (p2 * p4 * p6 == 0) & (p4 * p6 * p8 == 0)
            else:
                m = c & (B >= 2) & (B <= 6) & (A == 1) & (p2 * p4 * p8 == 0) & (p2 * p6 * p8 == 0)
            if m.any():
                im[1:-1, 1:-1][m] = 0; changed = True
        if not changed:
            return im[1:-1, 1:-1].astype(bool)

def graph(sk):
    """Đồ thị 8 lân cận trên các pixel đường tâm; trọng số = độ dài cạnh (1 hoặc √2)."""
    ys, xs = np.nonzero(sk)
    idx = -np.ones(sk.shape, int); idx[ys, xs] = np.arange(len(ys))
    r, c, w = [], [], []
    for dy, dx in ((0, 1), (1, 0), (1, 1), (1, -1)):
        y2, x2 = ys + dy, xs + dx
        ok = (y2 >= 0) & (y2 < sk.shape[0]) & (x2 >= 0) & (x2 < sk.shape[1])
        j = np.full(len(ys), -1); j[ok] = idx[y2[ok], x2[ok]]; k = j >= 0
        r += list(np.nonzero(k)[0]); c += list(j[k]); w += [np.hypot(dy, dx)] * int(k.sum())
    G = sparse.coo_matrix((w, (r, c)), shape=(len(ys), len(ys))).tocsr()
    return ys, xs, G + G.T

def depth_map(f):
    im = np.asarray(Image.open(f), float)
    hand = hand_roi(im); mk = (np.asarray(Image.open(DS / "masks" / f.name)) > 0) & hand
    dt = ndi.distance_transform_edt(mk)
    sk = thin(mk)
    lab, nl = ndi.label(sk, structure=np.ones((3, 3)))
    sk = np.isin(lab, 1 + np.nonzero(ndi.sum(sk, lab, range(1, nl + 1)) >= MIN_COMP)[0])
    ys, xs, G = graph(sk)
    cx, cy = directions(mk)
    meas_i, prof = [], []
    for i, (y, x) in enumerate(zip(ys, xs)):
        o = profile(im, mk, hand, y, x, cx[y, x], cy[y, x])
        if o is not None:
            meas_i.append(i); prof.append(o[0])
    meas_i = np.array(meas_i, int)
    raw = np.full(len(ys), np.nan); raw[meas_i] = np.mean([outputs(m, np.array(prof))[:, 0] for m in HETERO], 0)
    _, comp = csgraph.connected_components(G, directed=False)
    good = np.isin(comp, np.unique(comp[meas_i]))                      # nhánh có ít nhất 1 điểm đo
    W = G.copy(); W.data = 1.0 / W.data
    Lap = sparse.diags(np.asarray(W.sum(1)).ravel()) - W
    known = ~np.isnan(raw); unk = good & ~known; val = raw.copy()
    if unk.any():                                                      # nội suy điều hòa (phương trình Laplace)
        val[unk] = spsolve(Lap[unk][:, unk].tocsc(), -Lap[unk][:, known] @ raw[known])
    val[~good] = np.nan
    deg = np.asarray(W.sum(1)).ravel(); sm = val.copy()
    for _ in range(SMOOTH_IT):                                         # làm trơn khuếch tán dọc mạch (~2 mm)
        sm = np.where(good, (sm + W @ np.where(good, sm, 0.0)) / (1 + deg), np.nan)
    dist = csgraph.dijkstra(G, directed=False, indices=meas_i, min_only=True) * MM_PER_PX
    return dict(im=im, hand=hand, ys=ys, xs=xs, G=G, raw=raw, depth=sm, good=good, interp=dist > GAP_MM,
                width=2 * dt[ys, xs] * MM_PER_PX, n_meas=len(meas_i), n=len(ys))

cnt = {}
for r in rows:
    cnt[r["image"]] = cnt.get(r["image"], 0) + 1
names = sorted(cnt)
pick = max(names[::25], key=lambda n: cnt[n])                          # ảnh ví dụ: nhiều điểm đo nhất trong mẫu
D = depth_map(DS / "images" / f"{pick}.png")
g = D["good"]
print(f"Ảnh 1: {D['n']} pixel đường tâm, giữ {int(g.sum())}, đo được {D['n_meas']}, nội suy {D['interp'][g].mean():.0%}; "
      f"trung vị {np.nanmedian(D['raw']):.2f} mm (đo) → {np.nanmedian(D['depth'][g]):.2f} mm (sau làm trơn)")

In [ ]:
# Ô 17 — Vẽ bản đồ 3D: ảnh tĩnh + bản xoay được (plotly trong khung iframe riêng; cần mạng để tải plotly.js)
import html as _html
from mpl_toolkits.mplot3d.art3d import Line3DCollection
VMIN, VMAX, s = 1.0, 4.0, MM_PER_PX
ys, xs, dep, it = D["ys"], D["xs"], D["depth"], D["interp"]
C = sparse.triu(D["G"]).tocoo(); keep = g[C.row] & g[C.col]; i, j = C.row[keep], C.col[keep]
X, Y = xs * s, ys * s
cmap, norm = plt.get_cmap("jet_r"), plt.Normalize(VMIN, VMAX)
fig = plt.figure(figsize=(16, 6.6)); a = fig.add_subplot(1, 2, 1, projection="3d")
for sel, alpha, lw in ((~(it[i] | it[j]), 1.0, 1.0), (it[i] | it[j], 0.35, 0.5)):
    ii, jj = i[sel], j[sel]
    segs = np.stack([np.c_[X[ii], Y[ii], -dep[ii]], np.c_[X[jj], Y[jj], -dep[jj]]], 1)
    a.add_collection(Line3DCollection(segs, colors=cmap(norm((dep[ii] + dep[jj]) / 2)),
                                      linewidths=lw * (0.8 + 0.8 * (D["width"][ii] + D["width"][jj])), alpha=alpha))
hc = np.argwhere(D["hand"] ^ ndi.binary_erosion(D["hand"]))
a.scatter(hc[::3, 1] * s, hc[::3, 0] * s, 0, s=0.6, color="0.55", depthshade=False)
x0, x1, y0, y1 = hc[:, 1].min() * s, hc[:, 1].max() * s, hc[:, 0].min() * s, hc[:, 0].max() * s
a.set_xlim(x0, x1); a.set_ylim(y1, y0); a.set_zlim(-5, 0); a.set_box_aspect((x1 - x0, y1 - y0, 30))
a.set_xlabel("X (mm)"); a.set_ylabel("Y (mm)"); a.set_zlabel("Độ sâu (mm)"); a.view_init(elev=28, azim=-62)
a.set_title("Bản đồ độ sâu tĩnh mạch 3D (dự đoán)")
b = fig.add_subplot(1, 2, 2)
b.imshow(np.where(D["hand"], D["im"], D["im"] * 0.3), cmap="gray", vmin=0, vmax=255)
b.scatter(xs[g & it], ys[g & it], c="white", s=1.2, alpha=0.6)
sc = b.scatter(xs[g & ~it], ys[g & ~it], c=dep[g & ~it], cmap="jet_r", vmin=VMIN, vmax=VMAX, s=2.5)
hy, hx = np.nonzero(D["hand"]); b.set_xlim(hx.min() - 5, hx.max() + 5); b.set_ylim(hy.max() + 5, hy.min() - 5); b.axis("off")
b.set_title("Nhìn từ trên xuống (trắng = đoạn nội suy)")
cb = fig.colorbar(sc, ax=[a, b], shrink=0.8, pad=0.02); cb.set_label("Độ sâu dưới da (mm)"); cb.ax.invert_yaxis()
PATH["real_figs"].mkdir(parents=True, exist_ok=True)
fig.savefig(PATH["real_figs"] / "map3d_0.png", dpi=130, bbox_inches="tight"); plt.show()

def trace(sel, width, opacity, name):
    xx, yy, zz, cc = [], [], [], []
    for p_, q_ in zip(i[sel], j[sel]):
        xx += [round(X[p_], 2), round(X[q_], 2), None]; yy += [round(Y[p_], 2), round(Y[q_], 2), None]
        zz += [round(-dep[p_], 3), round(-dep[q_], 3), None]; cc += [round(dep[p_], 3), round(dep[q_], 3), round(dep[q_], 3)]
    return {"type": "scatter3d", "mode": "lines", "name": name, "x": xx, "y": yy, "z": zz, "opacity": opacity,
            "line": {"width": width, "color": cc, "colorscale": "Jet", "reversescale": True, "cmin": VMIN, "cmax": VMAX,
                     "showscale": name == "đo được", "colorbar": {"title": {"text": "Độ sâu (mm)"}}},
            "hovertemplate": "độ sâu %{line.color:.2f} mm<extra></extra>"}
meas = ~(it[i] | it[j])
hc4 = np.argwhere(D["hand"] ^ ndi.binary_erosion(D["hand"]))[::4]
hc4 = hc4[np.argsort(np.arctan2(hc4[:, 0] - hc4[:, 0].mean(), hc4[:, 1] - hc4[:, 1].mean()))]
traces = [trace(meas, 7, 1.0, "đo được"), trace(~meas, 3, 0.35, "nội suy"),
          {"type": "scatter3d", "mode": "lines", "name": "viền bàn tay", "x": list(np.round(hc4[:, 1] * s, 1)),
           "y": list(np.round(hc4[:, 0] * s, 1)), "z": [0] * len(hc4), "line": {"color": "#9AA5AD", "width": 2}, "hoverinfo": "skip"}]
layout = {"margin": {"l": 0, "r": 0, "t": 10, "b": 0}, "legend": {"x": 0.01, "y": 0.98},
          "scene": {"xaxis": {"title": "X (mm)"}, "yaxis": {"title": "Y (mm)", "autorange": "reversed"},
                    "zaxis": {"title": "Độ sâu (mm)", "range": [-5, 0]}, "aspectmode": "manual",
                    "aspectratio": {"x": 1, "y": float(np.ptp(Y) / np.ptp(X)), "z": 0.35},
                    "camera": {"eye": {"x": -1.2, "y": 1.5, "z": 0.9}}}}
page = ("<!doctype html><html><head><meta charset='utf-8'>"
        "<script src='https://cdnjs.cloudflare.com/ajax/libs/plotly.js/2.27.0/plotly.min.js'></script></head>"
        "<body style='margin:0'><div id='plot' style='width:100%;height:600px'></div><script>"
        f"Plotly.newPlot('plot', {json.dumps(traces, ensure_ascii=False, default=float)}, {json.dumps(layout)}, "
        "{responsive: true, displaylogo: false});</script></body></html>")
(PATH["real_figs"] / "map3d_0.html").write_text(page, encoding="utf-8")
display(HTML(f'<iframe srcdoc="{_html.escape(page, quote=True)}" style="width:100%;height:620px;border:0"></iframe>'))
print("Mở riêng bằng trình duyệt:", (PATH["real_figs"] / "map3d_0.html").as_uri())

## 4. Nơi lưu kết quả

In [ ]:
# Ô 18 — Các file kết quả ảnh tay thật vừa tạo (không commit, không chia sẻ — .gitignore đã chặn)
for folder in (PATH["real_out"], PATH["real_figs"]):
    files = sorted(q.name for q in folder.glob("*")) if folder.exists() else []
    print(folder, f"({len(files)} file)")
    for n in files:
        print("   ", n)